# LeetCode 2716 — Minimize String Length bằng Python

Link đề: https://leetcode.com/problems/minimize-string-length/ · Độ khó: **Easy** · Chủ đề: Hash Table, String

Skeleton LeetCode:

```python
class Solution:
    def minimizedStringLength(self, s: str) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: một thao tác chọn vị trí `i`, rồi xoá **ký tự giống `s[i]` gần nhất bên trái** và **gần nhất bên phải** (nếu có). Tìm độ dài **nhỏ nhất** sau khi thực hiện thao tác bao nhiêu lần tuỳ ý.
2. Đọc ra **bất biến** của thao tác: ký tự `s[i]` **không bị xoá**, các ký tự bị xoá đều **giống** `s[i]` ⇒ mỗi loại ký tự **không bao giờ biến mất hẳn**.
3. Từ đó suy ra đáp án = **số ký tự khác nhau** (distinct) trong `s`.
4. Dùng `set`, mảng đánh dấu 26 ô (`ord`), và **bitmask** để đếm ký tự khác nhau.
5. Viết 4 cách: mô phỏng đúng thao tác của đề (baseline), `len(set(s))`, mảng `seen[26]`, bitmask.
6. Kiểm chứng bằng **vét cạn mọi dãy thao tác** trên chuỗi ngắn và test ngẫu nhiên có seed.
7. Tránh các lỗi quen thuộc: nghĩ rằng xoá được hết ký tự lặp, chỉ xét ký tự kề nhau, đếm sai số ký tự xuất hiện một lần.

## 2. Đề bài & ví dụ

Cho chuỗi `s` (chỉ chữ cái thường), đánh chỉ số từ `0`. Bạn được thực hiện thao tác sau **bất kỳ số lần nào** (kể cả không lần nào):

- Chọn một chỉ số `i` trong `s`; gọi `c = s[i]`.
- **Xoá** ký tự `c` **gần nhất bên trái** `i` (nếu có).
- **Xoá** ký tự `c` **gần nhất bên phải** `i` (nếu có).

Trả về **độ dài nhỏ nhất** có thể đạt của `s`.

| Ví dụ | s | Output | Giải thích |
|---|---|---|---|
| 1 | `"aaabc"` | `3` | chọn `i = 1` (`a`): xoá `a` ở `0` và `a` ở `2` ⇒ `"abc"`, độ dài 3 |
| 2 | `"cbbd"` | `3` | chọn `i = 1` (`b`): không có `b` bên trái, xoá `b` ở `2` ⇒ `"cbd"` |
| 3 | `"baadccab"` | `4` | chọn nhiều lần ⇒ còn `"badc"` |

Constraints: `1 <= s.length <= 100`; `s` chỉ gồm chữ cái thường.

In [1]:
# Ví dụ 1: s = "aaabc", chọn i = 1
s = "aaabc"
i = 1
c = s[i]
left = s.rfind(c, 0, i)         # vị trí c gần nhất bên trái i (hoặc -1)
right = s.find(c, i + 1)        # vị trí c gần nhất bên phải i (hoặc -1)
print("c =", c, "| gần nhất trái:", left, "| gần nhất phải:", right)
remain = "".join(ch for j, ch in enumerate(s) if j not in (left, right))
print("sau khi xoá:", remain, "độ dài", len(remain))

c = a | gần nhất trái: 0 | gần nhất phải: 2
sau khi xoá: abc độ dài 3


## 3. Kiến thức nền

### 3.1. Tìm vị trí ký tự: `find`, `rfind`

- `s.find(c, start)` trả về chỉ số `c` đầu tiên từ `start` trở đi, hoặc `-1` nếu không có.
- `s.rfind(c, 0, i)` trả về chỉ số `c` **cuối cùng** trong đoạn `s[0:i]` (tức gần `i` nhất ở bên trái), hoặc `-1`.

Hai hàm này cho ta "ký tự giống `c` gần nhất bên phải / bên trái".

In [2]:
s = "baadccab"
print(s.find("a", 0), s.find("a", 3), s.find("z"))      # 1 6 -1
print(s.rfind("a", 0, 6), s.rfind("a", 0, 1))            # 2 -1
print(s.count("a"), s.count("d"))                         # 3 1

1 6 -1
2 -1
3 1


### 3.2. `set` (tập hợp): loại trùng và đếm

`set(s)` biến chuỗi thành **tập các ký tự khác nhau** (bỏ trùng, không có thứ tự). `len(set(s))` là số ký tự khác nhau.

In [3]:
s = "baadccab"
st = set(s)
print(st)                  # {'a', 'b', 'c', 'd'} (thứ tự có thể khác)
print(len(st))             # 4
print("a" in st, "z" in st)

{'c', 'b', 'a', 'd'}
4
True False


### 3.3. `ord` / `chr` và mảng đánh dấu 26 ô

`ord("a") = 97`, nên `ord(c) - ord("a")` đổi chữ thường sang chỉ số `0..25`. Ta có thể dùng list 26 ô `seen = [False] * 26` làm "bảng tra".

In [4]:
print(ord("a"), ord("z"), ord("d") - ord("a"))    # 97 122 3
print(chr(97 + 3))                                  # d
seen = [False] * 26
seen[ord("d") - ord("a")] = True
print(seen.index(True), sum(seen))                  # 3 1

97 122 3
d
3 1


### 3.4. Bitmask: dùng các bit của một số nguyên làm "bảng 26 công tắc"

Số nguyên có thể xem như dãy bit. `1 << k` là số có đúng bit thứ `k` bằng 1. `mask |= 1 << k` **bật** bit `k`. Đếm bao nhiêu bit đang bật bằng `bin(mask).count("1")`. Ta dùng bit `k` ứng với chữ cái thứ `k`.

In [5]:
mask = 0
for c in "baad":
    mask |= 1 << (ord(c) - ord("a"))
print(bin(mask))                     # 0b1011  (bit 0:a, 1:b, 3:d)
print(bin(mask).count("1"))          # 3
print(1 << 3, 5 | 2, bin(5 | 2))     # 8 7 0b111

0b1011
3
8 7 0b111


### 3.5. List: `del`, `.count`, `enumerate`

- `del a[j]` xoá phần tử tại chỉ số `j` (các phần tử sau dịch lên). Khi xoá nhiều chỉ số phải xoá **từ phải sang trái** để chỉ số bên trái không bị dịch.
- `a.count(c)` đếm số lần `c` xuất hiện trong list.
- `enumerate(a)` cho cặp `(chỉ số, phần tử)`.

In [6]:
a = list("baadcca")
del a[5], a[2]          # xoá chỉ số 5 rồi chỉ số 2 (từ phải sang trái nếu viết thành hai lệnh riêng)
print(a)                 # ['b', 'a', 'd', 'c', 'a']
a = list("baadcca")
for j in sorted([2, 5], reverse=True):
    del a[j]
print("".join(a), a.count("a"))
for i, c in enumerate("abc"):
    print(i, c, end=" | ")
print()

['b', 'a', 'd', 'c', 'a']
badca 2
0 a | 1 b | 2 c | 


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Đọc kỹ thao tác, ta thấy ba điều:

1. Ký tự **bị xoá** luôn **giống** `s[i]` (cùng chữ cái).
2. Chính `s[i]` **không bị xoá**.
3. Nếu `s[i]` là chữ cái duy nhất của nó trong chuỗi thì thao tác **không xoá gì**.

Hệ quả: chữ cái nào đã xuất hiện trong `s` thì **mãi mãi còn ít nhất một bản** (ví dụ chữ `a` bị xoá nhiều bản thì bản ở `s[i]` luôn còn). Vậy độ dài cuối **không thể nhỏ hơn số chữ cái khác nhau** của `s`.

Ngược lại, chữ cái nào còn từ 2 bản trở lên thì ta chọn `i` là một bản của nó và xoá bớt, cho tới khi còn **đúng 1 bản**. Làm cho từng chữ cái ⇒ đạt đúng **số chữ cái khác nhau**.

Tính tay `"baadccab"`: các chữ khác nhau: `b, a, d, c` ⇒ **4** (khớp ví dụ 3).

In [7]:
for s in ["aaabc", "cbbd", "baadccab"]:
    print(s, "-> các chữ khác nhau:", sorted(set(s)), "-> đáp án", len(set(s)))

aaabc -> các chữ khác nhau: ['a', 'b', 'c'] -> đáp án 3
cbbd -> các chữ khác nhau: ['b', 'c', 'd'] -> đáp án 3
baadccab -> các chữ khác nhau: ['a', 'b', 'c', 'd'] -> đáp án 4


## 5. Cách 1 — Brute Force / baseline: mô phỏng đúng thao tác của đề

**Ý tưởng:** làm **đúng như đề**. Lặp: tìm một chỉ số `i` mà ký tự `s[i]` còn **ít nhất một bản khác** trong chuỗi (nếu không có thì không còn gì để xoá, dừng); thực hiện thao tác tại `i` (xoá bản gần nhất bên trái và bên phải nếu có). Cuối cùng trả `len`.

Ta chọn `i` là **chỉ số nhỏ nhất** thoả điều kiện cho dễ cài đặt — chưa chứng minh là tối ưu, nhưng ở mục 12 ta sẽ đối chiếu với vét cạn mọi cách chọn.

In [8]:
def apply_op(a: list, i: int) -> list:
    """Áp dụng thao tác của đề lên list a tại chỉ số i, trả list mới."""
    c = a[i]
    left = max((j for j in range(i) if a[j] == c), default=None)            # gần nhất bên trái
    right = min((j for j in range(i + 1, len(a)) if a[j] == c), default=None)  # gần nhất bên phải
    kill = {x for x in (left, right) if x is not None}
    return [ch for j, ch in enumerate(a) if j not in kill]

def minimized_simulate(s: str) -> int:
    a = list(s)
    while True:
        pos = None
        for i, c in enumerate(a):
            if a.count(c) >= 2:           # c còn ít nhất một bản khác -> thao tác tại i sẽ xoá được
                pos = i
                break
        if pos is None:
            return len(a)
        a = apply_op(a, pos)

print(minimized_simulate("aaabc"))        # 3
print(minimized_simulate("cbbd"))         # 3
print(minimized_simulate("baadccab"))     # 4

3
3
4


**Độ phức tạp:** mỗi thao tác xoá ít nhất 1 ký tự nên có tối đa `n` thao tác; mỗi lần tìm `pos` dùng `count` lặp `O(n²)` trong trường hợp xấu. Tổng khoảng `O(n³)` (với `n ≤ 100` vẫn nhanh). Space `O(n)`.

**Nút thắt:** ta tốn rất nhiều công để mô phỏng, trong khi kết quả cuối chỉ phụ thuộc "mỗi chữ cái còn đúng một bản". Mục 4 đã gợi ý: chỉ cần **đếm số chữ cái khác nhau**.

## 6. Ý tưởng thuật toán / bước đột phá

**Phát biểu:** đáp án = **số ký tự khác nhau** trong `s`.

- *Cận dưới:* thao tác không bao giờ xoá `s[i]`, mà chỉ xoá bản **cùng chữ cái** với nó. Do đó mỗi chữ cái có mặt ban đầu vẫn có mặt cuối cùng ⇒ độ dài `≥` số chữ cái khác nhau.
- *Đạt được:* với mỗi chữ cái còn ≥ 2 bản, chọn `i` là một bản và xoá bớt (ít nhất một bản bị xoá); lặp lại tới khi còn 1 bản ⇒ độ dài `=` số chữ cái khác nhau.

Việc còn lại: đếm số chữ cái khác nhau nhanh nhất có thể (set, mảng 26 ô, bitmask).

In [9]:
# Kiểm chứng ý tưởng: cách mô phỏng và cách đếm chữ khác nhau cho cùng đáp án
for s in ["aaabc", "cbbd", "baadccab", "abcabcabc", "zzzz", "a"]:
    print(f"{s:<12} mô phỏng = {minimized_simulate(s)}, distinct = {len(set(s))}")
    assert minimized_simulate(s) == len(set(s))

aaabc        mô phỏng = 3, distinct = 3
cbbd         mô phỏng = 3, distinct = 3
baadccab     mô phỏng = 4, distinct = 4
abcabcabc    mô phỏng = 3, distinct = 3
zzzz         mô phỏng = 1, distinct = 1
a            mô phỏng = 1, distinct = 1


## 7. Các cách tiếp theo

Ngoài Cách 1 (mô phỏng), ta xem 3 cách đếm chữ khác nhau: (2) `set`, (3) mảng `seen[26]`, (4) bitmask.

### Cách 2 — `len(set(s))`

Giải thích: `set(s)` duyệt chuỗi, bỏ trùng ⇒ tập chữ khác nhau; `len` là số phần tử. Một dòng.

In [10]:
def minimized_set(s: str) -> int:
    return len(set(s))

print(minimized_set("aaabc"), minimized_set("cbbd"), minimized_set("baadccab"))

3 3 4


In [11]:
class SolutionSet:
    def minimizedStringLength(self, s: str) -> int:
        return len(set(s))

print(SolutionSet().minimizedStringLength("cbbd"))

3


**Độ phức tạp:** Time `O(n)` (mỗi ký tự băm một lần, trung bình `O(1)`), Space `O(k)` với `k ≤ 26` là số chữ khác nhau (ở đây tối đa 26 ⇒ coi như `O(1)`).

### Cách 3 — Mảng `seen` 26 ô

Không cần `set`: bảng 26 ô `bool`. Đọc từng ký tự `c`, tính `idx = ord(c) - ord("a")`; nếu `seen[idx]` còn `False` thì ta thấy chữ này **lần đầu**: bật `True` và tăng `count`.

In [12]:
def minimized_seen(s: str) -> int:
    seen = [False] * 26
    count = 0
    for c in s:
        idx = ord(c) - ord("a")
        if not seen[idx]:
            seen[idx] = True
            count += 1
    return count

print(minimized_seen("aaabc"), minimized_seen("cbbd"), minimized_seen("baadccab"))

3 3 4


In [13]:
class SolutionSeen:
    def minimizedStringLength(self, s: str) -> int:
        seen = [False] * 26
        count = 0
        for c in s:
            idx = ord(c) - ord("a")
            if not seen[idx]:
                seen[idx] = True
                count += 1
        return count

print(SolutionSeen().minimizedStringLength("baadccab"))

4


**Độ phức tạp:** Time `O(n)`, Space `O(26) = O(1)`.

### Cách 4 — Bitmask

Cùng ý như mảng 26 ô nhưng gói 26 "công tắc" vào **một số nguyên**: `mask |= 1 << (ord(c) - 97)`. Cuối cùng đếm bit 1: `bin(mask).count("1")` (Python 3.10+ có `mask.bit_count()`). Cách này rất hay gặp trong lập trình thi đấu vì tiết kiệm bộ nhớ và phép toán nhanh.

In [14]:
def minimized_mask(s: str) -> int:
    mask = 0
    for c in s:
        mask |= 1 << (ord(c) - ord("a"))
    return bin(mask).count("1")

print(minimized_mask("aaabc"), minimized_mask("cbbd"), minimized_mask("baadccab"))

3 3 4


In [15]:
class SolutionMask:
    def minimizedStringLength(self, s: str) -> int:
        mask = 0
        for c in s:
            mask |= 1 << (ord(c) - ord("a"))
        return bin(mask).count("1")

print(SolutionMask().minimizedStringLength("aaabc"))

3


**Độ phức tạp:** Time `O(n)`, Space `O(1)` (một số nguyên 26 bit; `bin(...)` tạo chuỗi tối đa 28 kí tự).

## 8. TRACE từng bước

Trace Cách 3 (mảng `seen`) với `s = "baadccab"`:

| i | `c` | `seen[c]` trước | hành động | `count` | tập đã thấy |
|---|---|---|---|---|---|
| 0 | b | False | bật, `count += 1` | 1 | {b} |
| 1 | a | False | bật, `count += 1` | 2 | {b, a} |
| 2 | a | True | bỏ qua | 2 | {b, a} |
| 3 | d | False | bật, `count += 1` | 3 | {b, a, d} |
| 4 | c | False | bật, `count += 1` | 4 | {b, a, d, c} |
| 5 | c | True | bỏ qua | 4 | |
| 6 | a | True | bỏ qua | 4 | |
| 7 | b | True | bỏ qua | 4 | |

Đáp án `4`.

**Trace thêm cho Cách 1 (mô phỏng):** với `"baadccab"`: (1) `i=0` (`b` còn bản ở 7) ⇒ xoá chỉ số 7: `"baadcca"`; (2) `i=1` (`a`, bản kề phải ở chỉ số 2) ⇒ `"badcca"`; (3) `i=1` ⇒ xoá `a` ở chỉ số 5 ⇒ `"badcc"`; (4) `i=3` (`c`) ⇒ xoá `c` ở 4 ⇒ `"badc"`. Dừng vì mọi chữ chỉ còn một bản.

In [16]:
def trace_seen(s):
    seen = [False] * 26
    count = 0
    for i, c in enumerate(s):
        idx = ord(c) - ord("a")
        if not seen[idx]:
            seen[idx] = True
            count += 1
            action = "lần đầu thấy -> bật"
        else:
            action = "đã thấy -> bỏ qua"
        print(f"i={i} c={c} idx={idx:>2} {action:<20} count={count}")
    print("đáp án =", count)
    return count

trace_seen("baadccab")

i=0 c=b idx= 1 lần đầu thấy -> bật  count=1
i=1 c=a idx= 0 lần đầu thấy -> bật  count=2
i=2 c=a idx= 0 đã thấy -> bỏ qua    count=2
i=3 c=d idx= 3 lần đầu thấy -> bật  count=3
i=4 c=c idx= 2 lần đầu thấy -> bật  count=4
i=5 c=c idx= 2 đã thấy -> bỏ qua    count=4
i=6 c=a idx= 0 đã thấy -> bỏ qua    count=4
i=7 c=b idx= 1 đã thấy -> bỏ qua    count=4
đáp án = 4


4

In [17]:
def trace_simulate(s):
    a = list(s)
    step = 0
    print(f"đầu: {''.join(a)}")
    while True:
        pos = next((i for i, c in enumerate(a) if a.count(c) >= 2), None)
        if pos is None:
            break
        step += 1
        c = a[pos]
        before = "".join(a)
        a = apply_op(a, pos)
        print(f"bước {step}: chọn i={pos} ({c}): {before} -> {''.join(a)}")
    print("còn lại:", "".join(a), "độ dài", len(a))
    return len(a)

trace_simulate("baadccab")
trace_simulate("aaabc")

đầu: baadccab
bước 1: chọn i=0 (b): baadccab -> baadcca
bước 2: chọn i=1 (a): baadcca -> badcca
bước 3: chọn i=1 (a): badcca -> badcc
bước 4: chọn i=3 (c): badcc -> badc
còn lại: badc độ dài 4
đầu: aaabc
bước 1: chọn i=0 (a): aaabc -> aabc
bước 2: chọn i=0 (a): aabc -> abc
còn lại: abc độ dài 3


3

## 9. Code nộp LeetCode

Dùng Cách 2 (`len(set(s))`) — ngắn nhất và đúng; ta kèm giải thích ở mục 10. Đúng class `Solution` và method `minimizedStringLength`.

In [18]:
class Solution:
    def minimizedStringLength(self, s: str) -> int:
        return len(set(s))

In [19]:
print(Solution().minimizedStringLength("aaabc"))       # 3
print(Solution().minimizedStringLength("cbbd"))        # 3
print(Solution().minimizedStringLength("baadccab"))    # 4

3
3
4


## 10. Vì sao thuật toán đúng

Gọi `D` là số chữ cái khác nhau của `s`. Ta chứng minh độ dài nhỏ nhất đúng bằng `D`.

**(a) Không thể nhỏ hơn `D`.** Một thao tác tại `i` chỉ xoá các ký tự **bằng `s[i]`** và **không xoá chính `s[i]`**. Xét chữ cái `x` bất kỳ đang có trong chuỗi. Giả sử một thao tác tại `i` xoá một bản `x` (tức `s[i] = x`): chính `s[i]` vẫn còn ⇒ `x` vẫn có mặt. Giả sử thao tác tại `i` có `s[i] ≠ x`: nó không động tới `x`. Vậy **sau mọi thao tác, tập chữ cái khác nhau không đổi** ⇒ độ dài luôn `≥ D`.

**(b) Luôn đạt được `D`.** Nếu còn chữ cái `x` xuất hiện ≥ 2 lần, chọn `i` là chỉ số của bản `x` đầu tiên. Vì có bản `x` khác nên ít nhất một trong hai phía (trái / phải) có bản `x` ⇒ thao tác xoá được ít nhất một ký tự. Mỗi thao tác như vậy giảm độ dài ít nhất `1`, độ dài luôn `≥ D` nên quá trình dừng sau hữu hạn bước, và khi dừng không chữ nào còn ≥ 2 bản ⇒ độ dài đúng bằng `D`.

Kết luận: độ dài nhỏ nhất `= D = len(set(s))`. Phần mô phỏng trong Cách 1 cũng là chứng minh "bằng chương trình" cho (b), và ở mục 12.4 ta vét cạn **mọi** thứ tự thao tác để xác nhận (a).

## 11. Phân tích Time / Space Complexity

Gọi `n = len(s)` (tối đa 100) và `σ = 26` là kích thước bảng chữ cái.

| Cách | Time | Space |
|---|---|---|
| 1. mô phỏng đúng đề | `O(n³)` (đếm lại trong mỗi bước) | `O(n)` |
| 2. `len(set(s))` | `O(n)` | `O(σ)` |
| 3. mảng `seen[26]` | `O(n)` | `O(σ)` |
| 4. bitmask | `O(n)` | `O(1)` |

Cận dưới là `O(n)` vì phải đọc mọi ký tự. `σ = 26` là hằng số nên Space của Cách 2, 3, 4 đều coi như `O(1)`.

## 12. Bộ test

### 12.1. Ví dụ của đề

In [20]:
sols = [SolutionSet(), SolutionSeen(), SolutionMask(), Solution()]
funcs = [minimized_simulate, minimized_set, minimized_seen, minimized_mask] + [x.minimizedStringLength for x in sols]

for f in funcs:
    assert f("aaabc") == 3
    assert f("cbbd") == 3
    assert f("baadccab") == 4
print("Ví dụ của đề: tất cả cách đều đúng")

Ví dụ của đề: tất cả cách đều đúng


### 12.2. Edge case

- Một ký tự: `"a"` → `1`.
- Toàn ký tự giống nhau: `"aaaa"` → `1`, `"z" * 100` → `1`.
- Mọi ký tự khác nhau: `"abcd"` → `4`; `"abcdefghijklmnopqrstuvwxyz"` → `26`.
- Ký tự lặp không kề nhau: `"abab"` → `2`, `"abcabc"` → `3`.
- Chuỗi dài 100: `"ab" * 50` → `2`; 26 chữ rồi lặp lại cho đủ 100 ký tự → `26`.

In [21]:
alpha = "abcdefghijklmnopqrstuvwxyz"
cases = {
    "a": 1, "aaaa": 1, "z" * 100: 1, "abcd": 4, alpha: 26,
    "abab": 2, "abcabc": 3, "ab" * 50: 2, (alpha * 4)[:100]: 26,
}
for f in funcs:
    for s, want in cases.items():
        assert f(s) == want, (f, s[:20])
print("Edge case OK:", len(cases), "trường hợp x", len(funcs), "cách")

Edge case OK: 9 trường hợp x 8 cách


### 12.3. Phản ví dụ cho cách làm sai

Các cách sai hay gặp ở mục 13: "xoá hết ký tự lặp", "chỉ xét ký tự kề nhau", "đếm ký tự xuất hiện đúng một lần". Ta xác nhận chúng cho kết quả khác đáp án đúng.

In [22]:
from collections import Counter

def sai_xoa_het_lap(s):
    return sum(1 for v in Counter(s).values() if v == 1)       # chỉ đếm ký tự xuất hiện đúng 1 lần

def sai_ke_nhau(s):
    out = []
    for c in s:
        if out and out[-1] == c:
            out.pop()
        else:
            out.append(c)
    return len(out)

for s in ["aaabc", "abab"]:
    print(s, "đúng:", Solution().minimizedStringLength(s), "| chỉ-xuất-hiện-1-lần:", sai_xoa_het_lap(s), "| kề-nhau:", sai_ke_nhau(s))
assert sai_xoa_het_lap("aaabc") != Solution().minimizedStringLength("aaabc")
assert sai_ke_nhau("abab") != Solution().minimizedStringLength("abab")

aaabc đúng: 3 | chỉ-xuất-hiện-1-lần: 2 | kề-nhau: 3
abab đúng: 2 | chỉ-xuất-hiện-1-lần: 0 | kề-nhau: 4


### 12.4. Vét cạn mọi dãy thao tác + đối chiếu ngẫu nhiên (có seed)

Oracle vét cạn: với chuỗi ngắn, xét **mọi trạng thái đạt được** bằng mọi dãy thao tác (tại mọi chỉ số `i` có thể), lấy độ dài nhỏ nhất. Vì không giả định chiến lược chọn `i` nào, đây là kiểm chứng độc lập cho kết luận "đáp án = số chữ khác nhau". Sau đó so sánh 7 cách trên chuỗi tuỳ ý hợp lệ.

In [23]:
import random, string
from functools import lru_cache

@lru_cache(maxsize=None)
def oracle_exhaustive(s: str) -> int:
    best = len(s)
    for i in range(len(s)):
        t = "".join(apply_op(list(s), i))
        if t != s:                              # thao tác có tác dụng mới đi tiếp (tránh vòng lặp vô hạn)
            best = min(best, oracle_exhaustive(t))
    return best

random.seed(2716)
for _ in range(300):
    n = random.randint(1, 9)
    alpha = random.choice(["ab", "abc", "abcd"])
    s = "".join(random.choice(alpha) for _ in range(n))
    want = oracle_exhaustive(s)
    assert want == len(set(s)), (s, want)
    for f in funcs:
        assert f(s) == want, (s, f)
print("OK: 300 chuỗi ngắn; vét cạn mọi dãy thao tác luôn bằng số chữ khác nhau")

OK: 300 chuỗi ngắn; vét cạn mọi dãy thao tác luôn bằng số chữ khác nhau


In [24]:
random.seed(27160)
for _ in range(2000):
    n = random.randint(1, 100)
    alpha = random.choice(["a", "ab", "abc", "abcdefgh", string.ascii_lowercase])
    s = "".join(random.choice(alpha) for _ in range(n))
    want = len(set(s))
    for f in funcs:
        assert f(s) == want, (s, f)
print("OK: 2000 chuỗi dài tới 100 ký tự; mọi cách khớp")

OK: 2000 chuỗi dài tới 100 ký tự; mọi cách khớp


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Nghĩ rằng xoá được **tất cả** ký tự lặp (đáp án = số ký tự xuất hiện đúng 1 lần)

In [25]:
from collections import Counter

def sai_chi_dem_xuat_hien_mot_lan(s: str) -> int:
    return sum(1 for v in Counter(s).values() if v == 1)

print("sai :", sai_chi_dem_xuat_hien_mot_lan("aaabc"))      # 2 (chỉ b, c)
print("đúng:", Solution().minimizedStringLength("aaabc"))   # 3 (còn một chữ a)
assert sai_chi_dem_xuat_hien_mot_lan("aaabc") != Solution().minimizedStringLength("aaabc")

sai : 2
đúng: 3


**Vì sao sai:** thao tác **không xoá** `s[i]`. Chữ `a` bị xoá hai bản ở chỉ số `0` và `2` nhưng bản `s[1]` vẫn ở lại. Mỗi chữ cái luôn còn ít nhất một bản ⇒ đếm **mọi chữ khác nhau**, không chỉ chữ xuất hiện một lần.

### Lỗi 2: Chỉ xoá ký tự **kề nhau** giống nhau (nhầm với bài 1047)

In [26]:
def sai_ke_nhau(s: str) -> int:
    out = []
    for c in s:
        if out and out[-1] == c:
            out.pop()                      # xoá cặp kề nhau giống nhau
        else:
            out.append(c)
    return len(out)

print("sai :", sai_ke_nhau("baadccab"))        # 4 (trùng ngẫu nhiên)
print("sai :", sai_ke_nhau("aaabc"))           # 3 (trùng ngẫu nhiên)
print("sai :", sai_ke_nhau("abab"))            # 4, đúng phải là 2
print("đúng:", Solution().minimizedStringLength("abab"))
assert sai_ke_nhau("abab") != Solution().minimizedStringLength("abab")

sai : 4
sai : 3
sai : 4
đúng: 2


**Vì sao sai:** thao tác của đề xoá bản **gần nhất** theo chỉ số — hai bản không cần kề nhau. Ví dụ `"abab"`: chọn `i = 0` (`a`) xoá `a` ở chỉ số `2` dù ở giữa có `b`; cuối cùng còn `"abb"`, rồi `"ab"`. Còn "xoá cặp kề nhau" là bài khác (LeetCode 1047) và dùng stack.

### Lỗi 3: Dùng `len(s) - số lần thao tác × 2` (nghĩ mỗi thao tác xoá đúng 2 ký tự)

In [27]:
def sai_moi_thao_tac_xoa_2(s: str) -> int:
    n = len(s)
    for v in Counter(s).values():
        n -= (v // 2) * 2         # SAI: giả sử bản lặp nào cũng xoá theo từng cặp
    return n

print("sai :", sai_moi_thao_tac_xoa_2("aaabc"))        # 5 - 2 = 3 (trùng đúng)
print("sai :", sai_moi_thao_tac_xoa_2("cbbd"))         # 4 - 2 = 2 (sai)
print("đúng:", Solution().minimizedStringLength("cbbd"))   # 3
assert sai_moi_thao_tac_xoa_2("cbbd") != Solution().minimizedStringLength("cbbd")

sai : 3
sai : 2
đúng: 3


**Vì sao sai:** một thao tác chỉ xoá **2** ký tự khi có bản cùng chữ ở **cả hai phía** của `i`. Nếu `i` là bản đầu hoặc cuối, chỉ xoá **1**. Quan trọng hơn, chữ cái còn một bản thì **dừng lại**, không xoá tiếp. Đừng suy đoán số lượng xoá; hãy dựa vào **bất biến**: mỗi chữ cái còn ít nhất một bản.

### Lỗi 4: Trả về `set` hoặc quên `len`

In [28]:
def sai_quen_len(s: str):
    return set(s)

r = sai_quen_len("aaabc")
print("sai :", r, type(r).__name__)
print("đúng:", Solution().minimizedStringLength("aaabc"), type(Solution().minimizedStringLength("aaabc")).__name__)
assert not isinstance(r, int)

sai : {'b', 'a', 'c'} set
đúng: 3 int


**Vì sao sai:** đề cần **độ dài** (số nguyên), không phải tập chữ cái. Kiểu trả về trong signature là `-> int`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Với đề "thao tác lạ, hỏi giá trị nhỏ nhất", hãy tìm **bất biến**: *cái gì không bao giờ mất?* ở đây là "mỗi loại chữ luôn còn ≥ 1 bản".
- Khi đề mô tả thao tác dài dòng mà ví dụ cho đáp án ≈ "số loại khác nhau" ⇒ nghĩ ngay đến `set` / `len(set(...))`.
- Khi ký tự thuộc bảng nhỏ (26 chữ thường): **mảng 26 ô** hoặc **bitmask** là cách chuẩn.
- Đếm bit bật: `bin(mask).count("1")`, hoặc `int.bit_count()` từ Python 3.10.
- Luôn tự kiểm bằng **vét cạn** trên input nhỏ để chắc chắn quy luật bạn đoán đúng (như mục 12.4).
- Cẩn thận phân biệt "xoá bản **gần nhất**" với "xoá bản **kề nhau**".

## 15. Biến thể của bài

1. **Trả về chuỗi cuối cùng** (mỗi chữ còn một bản, giữ thứ tự xuất hiện đầu tiên hoặc cuối cùng): `"".join(dict.fromkeys(s))` giữ thứ tự xuất hiện đầu.
2. **Số thao tác tối thiểu** để đạt độ dài nhỏ nhất: mỗi chữ cái xuất hiện `v` lần cần `ceil((v - 1) / 2)` thao tác (một thao tác ở bản giữa xoá được 2 bản).
3. **Số ký tự khác nhau trong mỗi cửa sổ** (sliding window): cần đếm bằng `dict`.
4. **Bài tương tự:** LeetCode 217 Contains Duplicate, 1832 Check if the Sentence Is Pangram (đủ 26 chữ), 389 Find the Difference.
5. **Chỉ đếm chữ xuất hiện đúng một lần** (LeetCode 387 First Unique Character): cần `Counter`.
6. **Số chữ khác nhau trong ba chuỗi** — bitmask OR của ba mask.

In [29]:
import math

def final_string(s):
    return "".join(dict.fromkeys(s))          # dict giữ thứ tự chèn (Python 3.7+)

def min_operations(s):
    return sum(math.ceil((v - 1) / 2) for v in Counter(s).values())

print(final_string("baadccab"))               # badc
print(min_operations("aaabc"), min_operations("baadccab"))    # 1  (a:3->1 | b:2->1 | a... )

badc
1 3


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. mô phỏng đúng đề | làm đúng thao tác, lặp đến hết | `O(n³)` | `O(n)` | hiểu đề, làm oracle |
| 2. `len(set(s))` | đếm chữ khác nhau bằng set | `O(n)` | `O(σ)` | **cách nộp bài khuyến nghị** |
| 3. mảng `seen[26]` | bảng 26 ô đánh dấu | `O(n)` | `O(σ)` | khi muốn tránh `set`, bảng chữ nhỏ |
| 4. bitmask | gói 26 công tắc trong một `int` | `O(n)` | `O(1)` | thi đấu, tối ưu bộ nhớ / tốc độ |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `len(set("hello"))`, `ord("c") - ord("a")`, `bin(1 << 3)`, `"banana".count("a")`.

**Bài 2 (tính tay).** `s = "abcabcab"`: sau thao tác tại `i = 3` (chữ `a`), chuỗi còn gì? Cuối cùng độ dài nhỏ nhất là bao nhiêu?

**Bài 3 (điền TODO).** Hoàn thành `minimized_todo(s)` bằng mảng `seen` 26 ô.

In [30]:
def minimized_todo(s):
    seen = [False] * 26
    count = 0
    for c in s:
        # TODO 1: idx = ord(c) - ord("a")
        # TODO 2: nếu seen[idx] còn False: bật True và count += 1
        pass
    # TODO 3: return count
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `minimized_mask2(s)` bằng bitmask mà không nhìn Cách 4. Kiểm tra với ba ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `min_operations(s)` đếm **số thao tác tối thiểu** để đạt độ dài nhỏ nhất (gợi ý: mỗi chữ cái xuất hiện `v` lần cần `ceil((v - 1) / 2)` thao tác — hãy tự kiểm tra bằng vét cạn nhỏ). (b) Viết `is_pangram(s)` (LeetCode 1832): `True` nếu `s` chứa đủ 26 chữ cái, bằng bitmask.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `"hello"` có `h, e, l, o` ⇒ `4`; `2`; `'0b1000'`; `3`. Bài 2: chữ `a` ở chỉ số 0, 3, 6; thao tác tại 3 xoá bản `a` ở 0 và 6.
- **Mức 2 (vừa):** Bài 3: `idx = ord(c) - ord("a")`, `if not seen[idx]: seen[idx] = True; count += 1`. Bài 4: `mask |= 1 << (ord(c) - 97)`.
- **Mức 3 (sâu):** Bài 5(a): thao tác tại bản **giữa** xoá 2 bản; với `v` bản cần giảm còn `1` tức xoá `v - 1` bản ⇒ `ceil((v - 1) / 2)` thao tác. Bài 5(b): `mask == (1 << 26) - 1`.

## 19. Đáp án tham khảo cho bài tập

In [31]:
import math
from collections import Counter

# Bài 1
assert len(set("hello")) == 4
assert ord("c") - ord("a") == 2
assert bin(1 << 3) == "0b1000"
assert "banana".count("a") == 3

# Bài 2: abcabcab, i = 3 (chữ a): xoá a gần nhất bên trái (chỉ số 0) và bên phải (chỉ số 6)
s = "abcabcab"
t = "".join(apply_op(list(s), 3))
assert t == "bcabcb"
assert len(set(s)) == 3 and Solution().minimizedStringLength(s) == 3

# Bài 3
def minimized_todo_ans(s):
    seen = [False] * 26
    count = 0
    for c in s:
        idx = ord(c) - ord("a")
        if not seen[idx]:
            seen[idx] = True
            count += 1
    return count

assert minimized_todo_ans("aaabc") == 3
assert minimized_todo_ans("cbbd") == 3

# Bài 4
def minimized_mask2(s):
    mask = 0
    for c in s:
        mask |= 1 << (ord(c) - 97)
    return bin(mask).count("1")

assert minimized_mask2("aaabc") == 3
assert minimized_mask2("cbbd") == 3
assert minimized_mask2("baadccab") == 4

# Bài 5
def min_operations(s):
    return sum(math.ceil((v - 1) / 2) for v in Counter(s).values())

def greedy_ops(s):
    """Số thao tác nhỏ nhất theo vét cạn (độ dài nhỏ nhất đạt được)."""
    from collections import deque
    start = s
    target = len(set(s))
    dist = {start: 0}
    dq = deque([start])
    while dq:
        cur = dq.popleft()
        if len(cur) == target:
            return dist[cur]
        for i in range(len(cur)):
            nxt = "".join(apply_op(list(cur), i))
            if nxt != cur and nxt not in dist:
                dist[nxt] = dist[cur] + 1
                dq.append(nxt)

for s in ["aaabc", "cbbd", "baadccab", "aaaaa", "abab", "abcabc"]:
    assert min_operations(s) == greedy_ops(s), (s, min_operations(s), greedy_ops(s))

def is_pangram(s):
    mask = 0
    for c in s:
        mask |= 1 << (ord(c) - 97)
    return mask == (1 << 26) - 1

assert is_pangram("thequickbrownfoxjumpsoverthelazydog") is True
assert is_pangram("leetcode") is False
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (đáp án = số chữ cái khác nhau)

```python
class Solution:
    def minimizedStringLength(self, s: str) -> int:
        return len(set(s))
```

**Checklist**

- [ ] Hiểu rằng thao tác **không bao giờ xoá `s[i]`** nên mỗi chữ cái luôn còn ít nhất một bản?
- [ ] Đáp án là số chữ cái **khác nhau** (không phải số chữ xuất hiện đúng một lần)?
- [ ] Không nhầm "bản gần nhất" với "bản kề nhau"?
- [ ] Trả về `len(...)` (số nguyên), không phải `set`?
- [ ] Đã thử `"a"`, `"aaaa"`, `"abab"`, chuỗi đủ 26 chữ và chuỗi dài 100?
- [ ] Nếu nghi ngờ quy luật: đã vét cạn trên input nhỏ để kiểm chứng?
- [ ] Signature đúng `minimizedStringLength(self, s: str) -> int`?